In [1]:
#load dataset
import pandas as pd
import numpy as np


df = pd.read_csv("E:/Inventory_managment/data/retail_store_inventory_cleaned.csv")
print(df.head())

         Date Store ID Product ID     Category Region  Inventory Level  \
0  2022-01-01     S001      P0001    Groceries  North              231   
1  2022-01-01     S001      P0002         Toys  South              204   
2  2022-01-01     S001      P0003         Toys   West              102   
3  2022-01-01     S001      P0004         Toys  North              469   
4  2022-01-01     S001      P0005  Electronics   East              166   

   Units Sold  Units Ordered  Demand Forecast  Price  Discount  \
0         127             55           135.47  33.50        20   
1         150             66           144.04  63.01        20   
2          65             51            74.02  27.99        10   
3          61            164            62.18  32.72        10   
4          14            135             9.26  73.64         0   

  Weather Condition  Holiday/Promotion  Competitor Pricing Seasonality  
0             Rainy                  0               29.69      Autumn  
1           

In [3]:
#Convert Date Column
df['date'] = pd.to_datetime(df['Date'])


In [4]:
#Sort Data
df = df.sort_values(by=['Product ID', 'Date'])

In [14]:
#Create Time base Feature
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['weekday'] = df['date'].dt.dayofweek

In [6]:
#Create Lag Features
df['lag_1'] = df.groupby('Product ID')['Units Sold'].shift(1)
df['lag_7'] = df.groupby('Product ID')['Units Sold'].shift(7)

In [8]:
#Rolling Mean Features
df['rolling_mean_7'] = (
    df.groupby('Product ID')['Units Sold']
    .rolling(7)
    .mean()
    .reset_index(0, drop=True)
)

df['rolling_mean_14'] = (
    df.groupby('Product ID')['Units Sold']
    .rolling(14)
    .mean()
    .reset_index(0, drop=True)
)


In [9]:
#Handle NaN Values by Lag
df.isnull().sum()
df.dropna(inplace=True)

In [11]:
#Encode Categorical Variables
df = pd.get_dummies(df, columns=['Category'], drop_first=True)


In [16]:
#final feature set
features = [
    'Inventory Level',
    'Price',
    'Discount',
    'day',
    'month',
    'weekday',
    'lag_1',
    'lag_7',
    'rolling_mean_7',
    'rolling_mean_14'
]

X = df[features]
y = df['Units Sold']


In [17]:
X.head()

,Inventory Level,Price,Discount,day,month,weekday,lag_1,lag_7,rolling_mean_7,rolling_mean_14
260,175,83.63,15,3,1,0,100.0,343.0,67.142857,115.714286
280,159,36.30,0,3,1,0,66.0,114.0,51.285714,106.857143
300,85,77.88,15,4,1,1,3.0,22.0,56.428571,103.571429
320,108,45.49,5,4,1,1,58.0,67.0,58.714286,99.000000
340,335,81.01,20,4,1,1,83.0,5.0,96.714286,99.000000


In [18]:
#saving Feature engineered data
final_df = pd.concat([X, y], axis=1)
final_df.to_csv("E:/Inventory_managment/data/retail_store_inventory_feature_engineered.csv", index=False)